# DART JAX denoiser TPU training run

This notebook trains the JAX latent denoiser on Kaggle TPU v5e-8 using the exported primitive shards. It is meant for the benchmark subset first: keep `ROOT` pointed at the same shard export used by the parity and partial VAE runs, then swap only `ROOT` when you are ready for the full-data export.

Upload/mount these inputs:

- the DART repo snapshot containing `jax_dart/`
- TPU shard dataset with `metadata.json`, `normalization.npz`, `shards/`, and text embeddings
- frozen JAX VAE checkpoint `latest.npz` or `checkpoint_*.npz`
- optional JAX denoiser checkpoint `latest.npz` if resuming
- optional SMPL-X locked-head directory if SMPL joint loss weights are enabled

In [ ]:
# Edit these paths for your Kaggle mount names.
REPO_IN = "/kaggle/input/datasets/markuskamsties/darttpu"
REPO = "/kaggle/working/DART"

# Keep this pointed at the same shard root used by the parity/benchmark runs.
ROOT = "/kaggle/input/datasets/markuskamsties/testrunning"

# Frozen VAE checkpoint from the successful VAE run.
VAE_NPZ = "/kaggle/input/datasets/markuskamsties/testrunning/latest.npz"

# Optional resume checkpoint for denoiser training. Leave empty for a fresh run.
DENOISER_INIT = ""

# Required only if --weight-smpl-joints-rec or --weight-joints-consistency is nonzero.
SMPL = "/kaggle/input/datasets/markuskamsties/testrunning/models_lockedhead"

OUT_SMOKE = "/kaggle/working/runs/jax_denoiser_tpu_smoke"
OUT_FULL = "/kaggle/working/runs/jax_denoiser_tpu_benchmark_subset_full"

import os
os.environ.update({
    "REPO_IN": REPO_IN,
    "REPO": REPO,
    "ROOT": ROOT,
    "VAE_NPZ": VAE_NPZ,
    "DENOISER_INIT": DENOISER_INIT,
    "SMPL": SMPL,
    "OUT_SMOKE": OUT_SMOKE,
    "OUT_FULL": OUT_FULL,
})

print("REPO_IN:", REPO_IN)
print("ROOT:", ROOT)
print("VAE_NPZ:", VAE_NPZ)
print("DENOISER_INIT:", DENOISER_INIT or "<fresh>")
print("OUT_FULL:", OUT_FULL)

## Copy repo to writable working space

In [ ]:
import os
from pathlib import Path

if not os.environ["REPO"].startswith("/kaggle/working/"):
    raise RuntimeError(f"REPO must be under /kaggle/working, got {os.environ['REPO']}")

os.chdir("/kaggle/working")
!rm -rf "$REPO"
!mkdir -p "$REPO"
!cp -a "$REPO_IN/." "$REPO/"
%cd /kaggle/working/DART
!ls -lah jax_dart/kaggle
!ls -lah jax_dart/training | grep denoiser || true
!ls -lah jax_dart/models | grep denoiser || true

## Code and shard guard

In [ ]:
import json, os
from pathlib import Path
import numpy as np

repo = Path(os.environ["REPO"])
required_files = [
    repo / "jax_dart/kaggle/denoiser_train_run.py",
    repo / "jax_dart/training/denoiser_train.py",
    repo / "jax_dart/models/mld_denoiser_state.py",
]
for path in required_files:
    print(path, "exists=", path.exists())
    if not path.exists():
        raise RuntimeError("The copied repo is stale. Upload/sync the updated DART repo and rerun copy-repo.")

root = Path(os.environ["ROOT"])
metadata = json.loads((root / "metadata.json").read_text())
arrays = metadata.get("arrays", {})
has_inline_text = "text_embedding" in arrays
has_text_table = (root / "text_embeddings.npz").exists()
has_text_ids = "text_id" in arrays
print("metadata_format:", metadata.get("format"))
print("num_shards:", metadata.get("export", {}).get("num_shards"))
print("num_samples:", metadata.get("export", {}).get("num_samples"))
print("primitive:", metadata.get("primitive", {}))
print("has_inline_text:", has_inline_text)
print("has_text_ids:", has_text_ids)
print("has_text_embeddings_npz:", has_text_table)
if not has_inline_text and not (has_text_ids and has_text_table):
    raise RuntimeError("Denoiser training needs text embeddings. Re-export shards with --text-mode inline or --text-mode ids.")

with np.load(os.environ["VAE_NPZ"]) as data:
    print("vae arrays:", len(data.files), "has_config=", "config_json" in data, "has_state=", any(k.startswith("state::") for k in data.files))

## Optional dependency refresh

In [ ]:
# Uncomment only if the Kaggle image does not already expose the expected TPU JAX stack.
# !pip install -q -U "jax[tpu]" flax optax orbax-checkpoint -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

## TPU/path check

In [ ]:
!python jax_dart/kaggle/denoiser_train_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --check-only \
  --root "$ROOT" \
  --vae-npz "$VAE_NPZ" \
  --out-dir "$OUT_FULL" \
  --smpl-model-dir "$SMPL"

## Short training smoke

In [ ]:
!python jax_dart/kaggle/denoiser_train_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --root "$ROOT" \
  --vae-npz "$VAE_NPZ" \
  --out-dir "$OUT_SMOKE" \
  --batch-samples 64 \
  --steps 20 \
  --log-every 1 \
  --eval-every 10 \
  --save-every 20 \
  --eval-batches 2 \
  --dtype bf16 \
  --model-type mlp \
  --h-dim 512 \
  --n-blocks 2 \
  --learning-rate 1e-4 \
  --weight-latent-rec 1.0 \
  --weight-feature-rec 1.0 \
  --weight-joints-delta 1e4 \
  --weight-transl-delta 1e4 \
  --weight-orient-delta 1e4 \
  --fail-on-nonfinite

## Full benchmark-subset run

In [ ]:
FULL_STEPS = 300_000
os.environ["FULL_STEPS"] = str(FULL_STEPS)

resume_args = ""
if os.environ.get("DENOISER_INIT"):
    resume_args = f"--init-npz {os.environ['DENOISER_INIT']}"
os.environ["RESUME_ARGS"] = resume_args

!python jax_dart/kaggle/denoiser_train_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --root "$ROOT" \
  --vae-npz "$VAE_NPZ" \
  $RESUME_ARGS \
  --out-dir "$OUT_FULL" \
  --batch-samples 512 \
  --steps "$FULL_STEPS" \
  --log-every 50 \
  --eval-every 5000 \
  --save-every 50000 \
  --eval-batches 16 \
  --dtype bf16 \
  --model-type mlp \
  --h-dim 512 \
  --n-blocks 2 \
  --cond-mask-prob 0.1 \
  --learning-rate 1e-4 \
  --anneal-lr \
  --grad-clip 1.0 \
  --weight-latent-rec 1.0 \
  --weight-feature-rec 1.0 \
  --weight-joints-delta 1e4 \
  --weight-transl-delta 1e4 \
  --weight-orient-delta 1e4 \
  --fail-on-nonfinite

## Inspect metrics

In [ ]:
import json, os
from pathlib import Path

metrics_path = Path(os.environ["OUT_FULL"]) / "metrics.jsonl"
print("metrics:", metrics_path, "exists=", metrics_path.exists())
if metrics_path.exists():
    rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    print("rows:", len(rows))
    for row in rows[-10:]:
        keys = ["step", "split", "loss", "latent_rec", "feature_rec", "steps_per_sec", "examples_per_sec", "elapsed_sec"]
        print({key: row.get(key) for key in keys if key in row})

## Package outputs

In [ ]:
!cd /kaggle/working && tar -czf dart_jax_denoiser_train_outputs.tgz runs